# Example 9.1 / 1: Map Keys and Data Columns

Companion notebook for **9.1 The `table` Statement** in the AMPL lab Documentation, section "Map Keys and Data Columns". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Map Keys and Data Columns

Each row has a unique `product` key and three numeric data columns:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                         # products, filled from the product column
param rate {PROD} > 0;            # tons produced per hour
param profit {PROD};              # profit per ton
param market {PROD} >= 0;         # maximum sales, tons

table Products IN "amplcsv" "products.csv":
    PROD <- [product], rate, profit, market;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
with open("products.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,6000\n"
            "coils,140,30,4000\n"
            "plate,160,29,3500\n")

In [5]:
# run.py
ampl.eval("load amplcsv.dll;")
ampl.read_table("Products")
print(ampl.set["PROD"].to_list())
print(ampl.get_data("rate", "profit", "market").to_pandas())

['bands', 'coils', 'plate']
       rate  profit  market
bands   200      25    6000
coils   140      30    4000
plate   160      29    3500


### Pure AMPL: model.run

`[product]` is the external key column, named after the file's heading. `PROD <- [product]` also
says to fill the AMPL set from those keys. The remaining columns supply `rate[p]`, `profit[p]`
and `market[p]` for each key `p`, matched by heading, so the column order in the file does not
matter.

Pure AMPL, with the same `products.csv`:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
with open("products.csv", "w") as f:
    f.write("product,rate,profit,market\n"
            "bands,200,25,6000\n"
            "coils,140,30,4000\n"
            "plate,160,29,3500\n")

In [8]:
%%ampl_eval
# model.run
load amplcsv.dll;
read table Products;
display PROD;
display rate, profit, market;

set PROD := bands coils plate;

:      rate profit market    :=
bands   200    25    6000
coils   140    30    4000
plate   160    29    3500
;

